# Knapsack Problem with AMPL in Python
## AMPL Modeling for Book Problem 2.12


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.


## Problem Statement

Book problem `2.12` is a binary knapsack problem: a thief chooses which products to steal so that the
total volume does not exceed `1.1 m^3` and the resale profit is maximized.

The student variation adds pair constraints: phones must travel together with speakers, laptops with
the sound system, and the microwave with the television.


In [ ]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

try:
    from amplpy import AMPL
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Install amplpy in the active environment before running this notebook.") from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


def create_ampl_instance(solver: str = "highs"):
    ampl = AMPL()
    ampl.eval(f"option solver {solver};")
    return ampl


ITEMS = ["phone_box", "laptop_dozen", "book_box", "cream_box", "sound_system", "car_battery", "microwave", "television", "speakers"]
VOLUME = {"phone_box": 0.17, "laptop_dozen": 0.5, "book_box": 0.32, "cream_box": 0.09, "sound_system": 0.2, "car_battery": 0.25, "microwave": 0.01, "television": 0.4, "speakers": 0.28}
PROFIT = {"phone_box": 2_000_000, "laptop_dozen": 6_000_000, "book_box": 900_000, "cream_box": 1_100_000, "sound_system": 900_000, "car_battery": 1_600_000, "microwave": 300_000, "television": 2_500_000, "speakers": 1_300_000}


@timed
def solve_knapsack_with_ampl(variation: bool = False, solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set I ordered;
        param v {I} >= 0;
        param u {I} >= 0;
        var x {I} binary;

        maximize TotalProfit:
            sum {i in I} u[i] * x[i];

        subject to Capacity:
            sum {i in I} v[i] * x[i] <= 1.1;
        '''
    )
    if variation:
        ampl.eval(
            r'''
            subject to PairPhonesSpeakers:
                x["phone_box"] = x["speakers"];

            subject to PairLaptopsSound:
                x["laptop_dozen"] = x["sound_system"];

            subject to PairMicrowaveTelevision:
                x["microwave"] = x["television"];
            '''
        )
    ampl.set["I"] = ITEMS
    ampl.param["v"] = VOLUME
    ampl.param["u"] = PROFIT
    ampl.solve(solver=solver)
    values = ampl.get_variable("x").get_values().to_dict()
    selected = [item for item in ITEMS if round(values[item]) == 1]
    return {
        "selected_items": selected,
        "total_profit": int(round(ampl.get_value("TotalProfit"))),
        "total_volume": round(sum(VOLUME[item] for item in selected), 2),
    }


## Base Model


In [ ]:
base_result, base_time = solve_knapsack_with_ampl(variation=False)
print("Base AMPL result:", base_result)
print(f"Elapsed time: {base_time:.6f} seconds")
assert base_result["selected_items"] == ["phone_box", "laptop_dozen", "cream_box", "car_battery", "microwave"]
assert base_result["total_profit"] == 11_000_000


## Student Variation


In [ ]:
variant_result, variant_time = solve_knapsack_with_ampl(variation=True)
print("Variation AMPL result:", variant_result)
print(f"Elapsed time: {variant_time:.6f} seconds")
assert variant_result["selected_items"] == ["laptop_dozen", "cream_box", "sound_system", "car_battery"]
assert variant_result["total_profit"] == 9_600_000
